# AI4ALL: YOLOv8 Advanced Model Training
This notebook handles the robust training of the Electronic Schematic Symbol Classifier, taking into account free Google Colab resource constraints (T4 GPU, limited RAM).


In [ ]:
!pip install ultralytics
import os
from ultralytics import YOLO

# Ensure data.yaml exists (Assuming the data pipeline has been run)
data_yaml_path = "/content/data_yolo/data.yaml"
print(f"Checking for {data_yaml_path}...")
if not os.path.exists(data_yaml_path):
    print("Warning: data.yaml not found! Make sure you ran the Data Pipeline notebook first to extract and format the dataset.")



## Phase 1: Robust Training Configuration
We are using early stopping (`patience=10`), a safe batch size (`batch=16`) to prevent OOM errors, and data augmentations suited for hand-drawn schematics (rotation, scale, and HSV adjustments).


In [ ]:
# Initialize the YOLOv8 Nano model
model = YOLO('yolov8n.pt')

# Advanced Training Loop
results = model.train(
    data=data_yaml_path,
    epochs=100,           # Train for up to 100 epochs
    patience=10,          # Early stopping if validation mAP doesn't improve for 10 epochs
    batch=16,             # Safe batch size for T4 GPU (use 8 if OOM occurs)
    imgsz=640,            # Standard image size
    device=0,             # Force GPU usage
    project='AI4ALL_Runs',# Directory to save results
    name='schematic_v1',  # Experiment name
    
    # --- Data Augmentation for Hand-Drawn Schematics ---
    degrees=15.0,         # Slight rotation (+/- 15 degrees)
    scale=0.5,            # Scale variations (zoom in/out)
    hsv_s=0.2,            # Saturation adjustments (handles different ink/paper)
    hsv_v=0.2,            # Value/Brightness adjustments
    fliplr=0.5,           # Horizontal flip (OK for circuits)
    flipud=0.0            # NO vertical flip (keeps symbols upright)
)

print("Training complete! Best weights saved to AI4ALL_Runs/schematic_v1/weights/best.pt")



## Phase 2: Model Evaluation
Generate metrics (mAP, Precision, Recall) on the validation set.


In [ ]:
# Evaluate the model on the validation set
metrics = model.val()

print(f"Mean Average Precision (mAP50-95): {metrics.box.map}")
print(f"Mean Average Precision (mAP50): {metrics.box.map50}")



## Phase 3: Inference & Export
Run a test inference and export the model to ONNX format for web deployment.


In [ ]:
# Export the model to ONNX format for faster inference in the web backend
export_path = model.export(format='onnx', imgsz=640)
print(f"Model exported successfully to: {export_path}")

# Optional: Run inference on a test image (replace 'path/to/test.jpg' with a real image)
# results = model('path/to/test.jpg', save=True)

